In [ ]:
"""Fine-tune GPT-4o mini on FSAE "exact rule text" task.
"""

import json, os, time, random
from pathlib import Path
from typing import List, Dict, Any, Tuple
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
#BASE_DIR = Path("")
SOURCE_JSONL = BASE_DIR / "rules_qa_dataset.jsonl"
WORK_DIR     = BASE_DIR / "finetune_artifacts"
WORK_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_JSONL  = WORK_DIR / "train2.jsonl"
VAL_JSONL    = WORK_DIR / "val2.jsonl"

BASE_MODEL   = "gpt-4o-mini-2024-07-18"

VAL_FRACTION = 0.98
MAX_TRAIN    = None
N_EPOCHS     = 2
LR_MULT      = 1.0
BATCH_SIZE   = "auto"

MODEL_SUFFIX = "fsae-exact-rule-text-Full"

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
client = OpenAI()

def to_messages_example(user_input: str, assistant_output: str) -> Dict[str, Any]:
    return {
        "messages": [
            {"role": "user", "content": user_input.strip()},
            {"role": "assistant", "content": assistant_output.strip()}
        ]
    }

def load_and_convert(src: Path) -> List[Dict[str, Any]]:
    rows = []
    with src.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            user_inp = obj.get("input", "")
            out = obj.get("output", "")
            if user_inp and out:
                rows.append(to_messages_example(user_inp, out))
    return rows

def train_val_split(examples: List[Dict[str, Any]], val_fraction: float = 0.0
                    ) -> Tuple[List[Dict[str, Any]], List[Dict[str, Any]]]:
    """
    If val_fraction <= 0 or dataset is tiny, return all to train and empty val.
    """
    random.shuffle(examples)
    n = len(examples)
    if val_fraction <= 0.0 or n < 2:
        return examples, []
    n_val = max(1, int(n * val_fraction))
    val = examples[:n_val]
    train = examples[n_val:]
    return train, val

def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def upload_file(path: Path, purpose: str = "fine-tune") -> str:
    with path.open("rb") as fh:
        file_obj = client.files.create(file=fh, purpose=purpose)
    return file_obj.id

def create_ft_job(train_file_id: str, val_file_id: str | None = None) -> str:
    kwargs = {
        "model": BASE_MODEL,
        "training_file": train_file_id,
        "suffix": MODEL_SUFFIX,
        "hyperparameters": {
            "n_epochs": N_EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate_multiplier": LR_MULT,
        },
    }
    if val_file_id is not None:
        kwargs["validation_file"] = val_file_id

    job = client.fine_tuning.jobs.create(**kwargs)
    print(f"[INFO] Created job: {job.id} (status={job.status})")
    return job.id

def wait_for_job(job_id: str, poll_seconds: int = 20) -> str:
    terminal = {"succeeded", "failed", "cancelled"}
    while True:
        job = client.fine_tuning.jobs.retrieve(job_id)
        print(f"[INFO] Job {job.id}: status={job.status}")
        if job.status in terminal:
            if job.status == "succeeded":
                print(f"[INFO] Fine-tuned model: {job.fine_tuned_model}")
                return job.fine_tuned_model
            raise RuntimeError(f"Fine-tune job ended with status {job.status}")
        time.sleep(poll_seconds)

def quick_infer(model_name: str, rule_id: str) -> str:
    user_prompt = (
        "What does rule {rid} state exactly? "
    ).format(rid=rule_id)

    resp = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": user_prompt}],
        temperature=0
    )
    return resp.choices[0].message.content

def main():
    assert SOURCE_JSONL.exists(), f"Missing dataset: {SOURCE_JSONL}"

    print("[1/7] Loading & converting dataset...")
    examples = load_and_convert(SOURCE_JSONL)
    if MAX_TRAIN is not None:
        examples = examples[:MAX_TRAIN]

    print(f"Total examples: {len(examples)}")
    train, val = train_val_split(examples, VAL_FRACTION)
    print(f"Train: {len(train)} | Val: {len(val)}")

    print("[2/7] Writing train/val JSONL...")
    write_jsonl(TRAIN_JSONL, train)
    if len(val) > 0:
        write_jsonl(VAL_JSONL, val)
    print(f"train -> {TRAIN_JSONL}")
    if len(val) > 0:
        print(f"val   -> {VAL_JSONL}")
    else:
        print("val   -> (skipped)")

    print("[3/7] Uploading files...")
    train_file_id = upload_file(TRAIN_JSONL, purpose="fine-tune")
    val_file_id   = upload_file(VAL_JSONL,   purpose="fine-tune") if len(val) > 0 else None
    print(f"train_file_id = {train_file_id}")
    print(f"val_file_id   = {val_file_id if val_file_id else '(none)'}")

    print("[4/7] Creating fine-tune job...")
    job_id = create_ft_job(train_file_id, val_file_id)

    print("[5/7] Monitoring job...")
    ft_model = wait_for_job(job_id, poll_seconds=30)

    print("[6/7] Test inference...")
    test_rule = "D.13.2.2"
    ans = quick_infer(ft_model, test_rule)
    print("\n=== SAMPLE OUTPUT ===")
    print(ans)

    print("\n[7/7] Done.")
    print(f"Fine-tuned model: {ft_model}")

if __name__ == "__main__":
    main()


[1/7] Loading & converting dataset...
Total examples: 1759
Train: 1759 | Val: 0
[2/7] Writing train/val JSONL...
train -> /home/knk22002/Downloads/Hackathon_ASME/FineTurningModel/finetune_artifacts/train2.jsonl
val   -> (skipped)
[3/7] Uploading files...
train_file_id = file-RGFNRbBnST5rxkqYpJiTRw
val_file_id   = (none)
[4/7] Creating fine-tune job...
[INFO] Created job: ftjob-f6Dpyn84Q0s9n7xS1yMTDrVO (status=validating_files)
[5/7] Monitoring job...
[INFO] Job ftjob-f6Dpyn84Q0s9n7xS1yMTDrVO: status=validating_files
[INFO] Job ftjob-f6Dpyn84Q0s9n7xS1yMTDrVO: status=validating_files
[INFO] Job ftjob-f6Dpyn84Q0s9n7xS1yMTDrVO: status=validating_files
[INFO] Job ftjob-f6Dpyn84Q0s9n7xS1yMTDrVO: status=validating_files
[INFO] Job ftjob-f6Dpyn84Q0s9n7xS1yMTDrVO: status=validating_files
[INFO] Job ftjob-f6Dpyn84Q0s9n7xS1yMTDrVO: status=validating_files
[INFO] Job ftjob-f6Dpyn84Q0s9n7xS1yMTDrVO: status=validating_files
[INFO] Job ftjob-f6Dpyn84Q0s9n7xS1yMTDrVO: status=validating_files
[INFO] Jo

In [ ]:
from openai import OpenAI
client = OpenAI()

#model_id = "ft:"

user_prompt = "We are a student engineering team designing a vehicle for the FSAE competition. Attached is the FSAE rules document. What does rule V.3.2.5 state exactly? Answer with only the text of the rule and no other words."

resp = client.chat.completions.create(
    model=model_id,
    messages=[
        # Optional but helpful to enforce formatting:
        {"role": "system", "content": "Answer with only the exact text of the requested rule. Do not include subrules."},
        {"role": "user", "content": user_prompt},
    ],
    temperature=0
)

print(resp.choices[0].message.content)


The following items are not allowed:
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or similar devices
• Any type of “safety wire” or “safety cable” or simi